# Lab 05 · A batch we can restart and explain

Automated Workflows in Satellite Image Processing · 5 November 2026

Lab 4 gave us one configured NDVI job. Lecture 5 asked whether its results were available, useful and explainable. Now we will run six requests and see what happens when one of them cannot finish.

By the end, you should be able to point to the expected work, explain an incomplete batch, recover it without losing history, and produce a report that another person can investigate.

The main exercise uses actual Sentinel-2 Collection 1 subsets from three dates near Kraków. Each date supplies two AOIs. The preserved inputs stay unchanged. Cloud screening comes later.

Core work: three new database tables, three logical repairs, two failure investigations, and a short decision. Allow about two hours. Coordinator interruption, missing outputs and the live acquisition route are extensions.

## 1. Open the package and inspect the evidence

Run the notebook from the Lab05 directory. Keep the supplied Python files beside it.

In [ ]:
from pathlib import Path
import json, sqlite3, subprocess, sys
import numpy as np
import rasterio
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
assert (ROOT / 'lab05_cli.py').is_file(), 'Open the notebook from the extracted package directory.'

def cli(*args, expected=(0,)):
    child = subprocess.run([sys.executable, str(ROOT / 'lab05_cli.py'), *map(str, args)], capture_output=True, text=True)
    assert child.returncode in expected, (child.returncode, child.stdout[-2500:], child.stderr[-2500:])
    if child.returncode == 87:
        return {'exit_code': 87, 'note': 'The isolated coordinator stopped abruptly.'}
    return json.loads(child.stdout)

def check(group='all'):
    child = subprocess.run([sys.executable, str(ROOT / 'check_lab05.py'), group], capture_output=True, text=True)
    print(child.stdout)
    return child.returncode == 0

def query(manifest_path, sql, params=()):
    import batch_plan
    manifest = batch_plan.load(manifest_path)
    with sqlite3.connect(manifest['_database']) as connection:
        connection.row_factory = sqlite3.Row
        connection.execute('PRAGMA foreign_keys=ON')
        return [dict(row) for row in connection.execute(sql, params)]

print('Python', sys.version.split()[0], 'Rasterio', rasterio.__version__)
fixture = json.loads((ROOT / 'fixtures/manifest.json').read_text())
print('Endpoint:', fixture['endpoint'], 'Collection:', fixture['collection'])
print('Requests:', len(fixture['members']))

In [ ]:
first_member = fixture['members'][0]
for role, entry in first_member['bands'].items():
    with rasterio.open(ROOT / 'fixtures' / entry['file']) as source:
        raw = source.read(1, masked=True)
        print(role, first_member['member_id'], source.shape, source.crs, source.scales, source.offsets, entry['sha256'])
        print('DN median:', float(np.ma.median(raw)))
print('Source transfer bytes:', sum(t['network_bytes'] for t in fixture['transfers']))

Name one property a checksum establishes and one scientific question it cannot answer. Explain why six requests here represent three dates rather than six independent acquisitions.

Your explanation:

*Write here.*

## 2. Design the new tables

Read `SCHEMA_CONTRACT.md`. Write all three tables in `student_batch_schema.sql`. The earlier acquisition and processing schemas are completed in `support/`.

Before writing SQL, sketch the links between a batch, its expected members, their requests, attempts and observations. Decide which records must exist when we know what we want but no calculation has started.

A member's planned request ID exists before its optional request foreign key can be filled. Explain why those are separate fields. Then run the schema check.

In [ ]:
schema_ok = check('schema')

Your explanation:

*Write here.*

## 3. Repair the coordination logic

The functions in `batch_repairs.py` execute, but the supplied versions make incorrect decisions. Read their contracts and repair them.

`classify_member` needs to distinguish blocked inputs, unresolved execution, verified completion and a permitted new attempt. A finished result may remain usable after its inputs move. An unresolved active execution needs attention even when an older result exists. The attempt budget does not reset when the coordinator restarts.

`verify_result` must reject evidence for another request, mismatching bytes, or missing verification. The adapter does the actual catalogue queries, hashing and raster inspection.

`summarize_batch` must count expected members once each using their newest observations. Repeated observations and joined locations do not create more requested work. Keep members that have no observation yet.

Predict which check will expose each supplied defect before editing.

In [ ]:
for group in ('classify', 'verify', 'summary'):
    check(group)

Your explanation:

*Write here.*

## 4. Register expected work and run the batch

Finish the schema and function repairs before creating processing evidence. The next cell imports the captured real inputs into a fresh catalogue and freezes membership. It does not make a processing attempt.

Rerunning the notebook keeps existing work. To begin a separate experiment, choose a new workspace name.

In [ ]:
assert check('all'), 'Finish the schema and three logical repairs first.'
MANIFEST = ROOT / 'work/batch.json'
if not MANIFEST.exists():
    print(cli('setup', '--directory', ROOT / 'work'))

expected_before = query(MANIFEST, 'SELECT member_id, planned_request_id, request_id FROM batch_members ORDER BY member_id')
expected_before

Write a query that lists expected members with no application attempt. Run it now. It should include the whole batch. Keep the query, its parameters and your interpretation in `investigation_queries.sql`.

In [ ]:
# Replace this starter query with your investigation query.
# no_attempt_sql = 
query(MANIFEST, 'SELECT member_id, planned_request_id FROM batch_members ORDER BY member_id')

In [ ]:
plan = cli('plan', '--manifest', MANIFEST)
print('Writes the database:', plan['writes_database'])
print([(m['member_id'], m['state'], m['reason']) for m in plan['members']])
result = cli('run', '--manifest', MANIFEST)
print('Complete:', result['complete'], 'New invocations:', len(result['invocations']))

Predict what a second invocation will change. Will it create another scientific product, another application attempt, or only new batch observations?

In [ ]:
attempts_before = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
repeat = cli('run', '--manifest', MANIFEST)
attempts_after = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
print('New invocations:', len(repeat['invocations']))
assert repeat['complete'] and not repeat['invocations']
assert attempts_before == attempts_after

Your explanation:

*Write here.*

## 5. Look at the results and their report

A contact sheet makes the six results easy to compare. The saved raster values are not clipped. The display uses a fixed interval; blue and magenta identify values outside it.

Export a report, then open its HTML file. Follow one count to its members and inspect a request, producing attempt and verification time. Scientific cloud suitability should remain unassessed.

In [ ]:
report_dir = ROOT / 'reports/main'
if not report_dir.exists():
    print(cli('report', '--manifest', MANIFEST, '--output', report_dir))
report = json.loads((report_dir / 'report.json').read_text())
print(report['summary'])
from IPython.display import Image, display
if (report_dir / 'preview.png').is_file():
    display(Image(filename=str(report_dir / 'preview.png')))
print('Open:', report_dir / 'index.html')

If the report says complete, what has it established? Give two reasons that still would not justify calling the image a clear sky vegetation observation. Identify the observation times separately from the report generation time.

Your explanation:

*Write here.*

## 6. Case A · One input disappears

The case helper creates a separate workspace, moves one red input aside, and runs the normal coordinator. Your main workspace and preserved fixtures stay intact.

Predict the expected member count, attempt count and number of completed requests. Then identify the exact blocked member and input role using SQL.

In [ ]:
def ensure_case(name, directory, expected=(0, 1)):
    case_manifest = ROOT / directory / 'batch.json'
    if not case_manifest.exists():
        result = cli('case', '--name', name, '--directory', ROOT / directory, expected=expected)
        print('Created:', name, result.get('manifest', result.get('note')))
    return case_manifest

missing = ensure_case('missing-input', 'cases/missing')
query(missing, """SELECT m.member_id, o.state, o.reason, o.observed_at
FROM batch_members m JOIN batch_observations o USING(batch_id,member_id)
WHERE o.observation_id=(SELECT MAX(x.observation_id) FROM batch_observations x
WHERE x.batch_id=m.batch_id AND x.member_id=m.member_id)
ORDER BY m.member_id""")

In [ ]:
query(missing, """SELECT a.artifact_id, json_extract(a.lineage_json,'$.asset_key') AS role,
 l.status, l.local_path, l.checked_at
FROM processing_artifacts a JOIN processing_artifact_locations l USING(artifact_id)
WHERE a.source_artifact_id IS NOT NULL AND l.status <> 'available'""")

Explain why the blocked member has no failed application attempt. How does expected membership let us see work that never started? Add the query that established your diagnosis to your query file.

Your explanation:

*Write here.*

In [ ]:
note = json.loads((missing.parent / 'case_note.json').read_text())
if Path(note['held']).exists() and not Path(note['original']).exists():
    print(cli('restore-input', '--manifest', missing))
recovered = cli('run', '--manifest', missing)
print('Complete after restoration:', recovered['complete'])
query(missing, 'SELECT status, COUNT(*) AS n FROM processing_attempts GROUP BY status')

## 7. Case B · The worker stops before publication

This helper waits for the actual child process to stop with exit code 86. The child wrote and validated a temporary raster, then exited without running cleanup. Its attempt still says running.

Predict what the coordinator will do with the remaining requests. We still permit only one active processing invocation per catalogue.

In [ ]:
worker = ensure_case('interrupted-worker', 'cases/worker')
query(worker, 'SELECT attempt_id, request_id, status, stage, temp_path, final_path FROM processing_attempts ORDER BY started_at')

Use SQL and file inspection to distinguish an interrupted execution from a missing input. Record why it is justified to assert that this child stopped. In a real unattended system, a timestamp alone would not establish that.

In [ ]:
running = query(worker, "SELECT attempt_id, temp_path, final_path FROM processing_attempts WHERE status='running'")
for row in running:
    print(row['attempt_id'], 'temporary:', Path(row['temp_path']).exists(), 'final:', Path(row['final_path']).exists())
print('Output links:', query(worker, 'SELECT COUNT(*) AS n FROM processing_outputs'))

Your explanation:

*Write here.*

In [ ]:
for row in running:
    print(cli('reconcile', '--manifest', worker, '--attempt-id', row['attempt_id'], '--confirm-stopped'))
worker_recovered = cli('run', '--manifest', worker)
assert worker_recovered['complete']
query(worker, 'SELECT request_id, status, error_code, started_at FROM processing_attempts ORDER BY started_at')

Write a query that finds a request with an interrupted or failed attempt and a later producing attempt. Show both identities. Explain why counting attempts would overstate delivered work.

In [ ]:
# Replace this starter with your recovery query.
query(worker, 'SELECT request_id, attempt_id, status FROM processing_attempts ORDER BY started_at')

## 8. A decision another person can follow

It is Saturday morning. One member is blocked because an input file is missing. Five other members have verified outputs. Delivery is required on Monday afternoon.

Write a short note stating what is known, what remains uncertain, the next check, and whether you would act now or defer. Explain how your decision changes if an incorrect calibration may have affected already published products.

Save your useful SQL with a question, parameters, empty-result interpretation and next action. Sketch one overview and one detail view using the fields you actually have. Mark any proposed future measurements as unavailable rather than inventing them.

Your explanation:

*Write here.*

## Extensions

1. Create `interrupted-coordinator` in a new case directory. The command exits with code 87 after one completed member. Run the same manifest again and prove that only five jobs launch.
2. Create `missing-output`. Inspect both the old succeeded attempt and the new location observations. Recover the output without rewriting history.
3. Add a second copy of a verified output, then remove one copy. Explain why a query using any missing location gives the wrong answer.
4. Repeat the explicit live acquisition command from `README.md` into a new directory. Compare its captured source metadata and subset hashes with the supplied manifest.
5. Change a scientific setting in one configuration and freeze a new manifest. Contrast that new request with a change to an output directory.

Keep extensions separate from your core evidence. Never remove old attempt rows to make a report look clean.

## Submit

Keep your notebook, three table definitions, three repairs, query file, one saved report, and the decision note. Include the failed/interrupted evidence as well as the recovered outcome.

The extensions are optional.